In [1]:
import random
from dataclasses import dataclass
from enum import Enum

from deap import base, creator, tools

from params.params import SensorType, Gene
from params.params import VALID_NODE_IDS, POPULATION_SIZE
from params.seeding import SEED_INDIVIDUALS

In [2]:
# Define the fitness to maximize
creator.create("FitnessMax", base.Fitness, weights=(1.0,))

# Define the Individual class as a list of Gene objects with an associated fitness.
creator.create("Individual", list, fitness=creator.FitnessMax)

In [3]:
def create_gene(assigned_node_id):
    """Generates a single active sensor configuration."""
    sensor_type = random.choice(list(SensorType))

    # Define physical rotation bounds.
    # Note: If you are using 360-degree spinning LiDARs, roll/pitch is sufficient.
    # If using directional solid-state LiDARs, you may need to swap 'roll' for 'yaw'.
    pitch = random.randint(-90, 90)
    roll = random.randint(-180, 180)

    return Gene(sensor_type=sensor_type, node_id=assigned_node_id, pitch=pitch, roll=roll)


def create_individual(icls):
    """Generates an individual with 1 to 4 active sensors."""
    num_sensors = random.randint(1, 4)

    # Ensure unique node IDs for each sensor in the individual
    unique_nodes = random.sample(VALID_NODE_IDS, num_sensors)

    genes = [create_gene(node_id) for node_id in unique_nodes]

    # Initialize the DEAP list class with our generated genes
    return icls(genes)


def create_seeded_population(icls, individual_factory, seed_contents, population_size, shuffle=False):
    """Builds a population from seeded individuals plus random fill."""
    seeded_population = [icls(seed) for seed in seed_contents]

    if len(seeded_population) > population_size:
        raise ValueError(
            f"seed_contents contains {len(seeded_population)} individuals, "
            f"but population_size is only {population_size}."
        )

    random_population = [individual_factory() for _ in range(population_size - len(seeded_population))]
    population = seeded_population + random_population

    if shuffle:
        random.shuffle(population)

    return population

In [13]:
toolbox = base.Toolbox()

# Register the individual generator
toolbox.register("individual", create_individual, creator.Individual)

# Register the population generator (creates a list of individuals)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# NOTE: Consider Demes
# A deme is a sub-population that is contained in a population.
# https://deap.readthedocs.io/en/master/tutorials/basic/part1.html#demes

# Create initial population
population = create_seeded_population(
    creator.Individual,
    toolbox.individual,
    SEED_INDIVIDUALS,
    POPULATION_SIZE,
)

In [14]:
from utils import visualization

visualization.visualize_population(population)

Individual,Sensor 1,Sensor 2,Sensor 3,Sensor 4
Individual 1,"LIDAR_16_CHnode 5pitch 0, roll 0",—,—,—
Individual 2,"SOLID_STATEnode 42pitch 10, roll -15","LIDAR_32_CHnode 99pitch -5, roll 20",—,—
Individual 3,"SOLID_STATEnode 28pitch 50, roll 99","LIDAR_32_CHnode 156pitch -36, roll 13","LIDAR_32_CHnode 133pitch -53, roll -135","LIDAR_16_CHnode 98pitch -22, roll 70"
Individual 4,"LIDAR_16_CHnode 40pitch 81, roll 98","LIDAR_32_CHnode 180pitch 13, roll -133",—,—
Individual 5,"LIDAR_16_CHnode 130pitch 79, roll -111","LIDAR_32_CHnode 168pitch -24, roll 68","LIDAR_32_CHnode 187pitch 57, roll 60",—
